# Interoperabilidad · Grafo de conexiones entre datasets

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Data-Observatory/datopia-notebooks/blob/main/notebooks/interoperabilidad/grafo/demo.ipynb)
[![Licencia](https://img.shields.io/badge/licencia-MIT-blue.svg)](../../../LICENSE)

---

## Descripción

Un grafo interactivo de **cómo se conectan los datasets de Datopia entre sí**: qué tablas se cruzan por qué columnas,
cuáles se pueden cruzar por su geometría y cuáles relaciones aún no existen pero ya se asumen.

Se construye con dos entradas, sin credenciales:

1. El **modelo de datos público** (`_meta/datamodel.json`): datasets, columnas, geometrías y las llaves foráneas declaradas.
2. `relaciones-objetivo.json`: las relaciones que se asumen cerradas (issues #56 a #59) más las que existen pero nadie ha
   declarado todavía.

### Reglas del grafo

| Tipo | Regla |
|---|---|
| **Tabular** | Dos datasets se relacionan solo si uno referencia al otro, o si **ambos referencian una misma tabla**. La tabla compartida es un nodo; no se dibujan líneas entre hermanos (la tarjeta de cada nodo lista con quién comparte tabla). |
| **Espacial** | Tener geometría es una **insignia** del nodo (● puntos, ∿ líneas, ⬠ polígonos). La capa espacial conecta cada dataset con geometría al polígono de referencia (las comunas): *dentro de* para puntos, *cruza* para líneas. Todos los pares espaciales posibles se pueden ver como capa teórica, apagada por defecto. |
| **Estado** | *Objetivo* asume cerrados #56 a #59 (códigos `cut_*` en todos lados, geometría en establecimientos, v2 fusionado en v1). *Hoy* muestra solo lo publicado. |

Línea punteada azul = relación objetivo; punteada ámbar con ⚠ = relación objetivo con un problema conocido (la relación teórica
existe igual); punteada violeta = fusión de datasets.

In [1]:
# @title Archivos auxiliares
import json, pathlib, sys, tempfile, urllib.request
from IPython.display import HTML, display

RAW = "https://raw.githubusercontent.com/Data-Observatory/datopia-notebooks/main/notebooks/interoperabilidad/grafo"
for nombre in ["grafo_datasets.py", "visor.html", "relaciones-objetivo.json"]:
    if not pathlib.Path(nombre).exists():  # en Colab los archivos de la carpeta no vienen con el notebook
        urllib.request.urlretrieve(f"{RAW}/{nombre}", nombre)
sys.path.insert(0, str(pathlib.Path(".").resolve()))
import grafo_datasets as gd
print("Archivos listos")

Archivos listos


---
## 1. Entradas

El modelo de datos público se descarga sin credenciales.

In [2]:
datamodel = gd.cargar_datamodel()
objetivo = json.loads(pathlib.Path("relaciones-objetivo.json").read_text(encoding="utf-8"))

print(f"{len(datamodel['datasets'])} datasets · {len(datamodel['relations'])} relaciones declaradas · "
      f"{len(datamodel['dataset_groups'])} grupos")
print(f"{len(objetivo['relaciones'])} relaciones objetivo adicionales")
for supuesto in objetivo["supuestos"]:
    print(" ·", supuesto)

35 datasets · 39 relaciones declaradas · 6 grupos
12 relaciones objetivo adicionales
 · #56: códigos de comuna, provincia y región con los nombres oficiales cut_com, cut_prov y cut_reg en todos los datasets
 · #57: sinca estaciones publica cut_com, cut_prov y cut_reg
 · #58: establecimientos de salud publica geometría Point (más lon y lat)
 · #59: division-territorial v2 se fusiona en v1; v1 queda con geometría


---
## 2. Construir el grafo

`gd.construir` junta las dos entradas, calcula las capas espaciales a partir de las geometrías declaradas y deja las
posiciones de los nodos fijas (misma entrada, mismo dibujo).

In [3]:
grafo = gd.construir(datamodel, objetivo)
resumen = gd.resumen(grafo)

print("Aristas por tipo y estado:")
for clave, n in resumen["aristas"].items():
    print(f"  {clave:22s} {n}")

for estado in ("hoy", "objetivo"):
    print(f"\nGrupos conectados por tablas ({estado}):")
    for n, nombres in resumen[f"componentes_{estado}"]:
        muestra = ", ".join(nombres[:4]) + (" …" if n > 4 else "")
        print(f"  {n:2d} datasets: {muestra}")

Aristas por tipo y estado:
  espacial/hoy           4
  espacial/objetivo      5
  espacial-par/hoy       10
  espacial-par/objetivo  15
  fusion/objetivo        1
  tabular/hoy            36
  tabular/objetivo       12

Grupos conectados por tablas (hoy):
  15 datasets: dtpm / gtfs-agency, dtpm / gtfs-calendar, dtpm / gtfs-calendar-dates, dtpm / gtfs-feeds …
   6 datasets: sinca / calidad, sinca / estaciones, sinca / estaciones-variables, sinca / medicion-diaria …
   4 datasets: ine / censos-poblacionales v1l, ine / censos-poblacionales v1w, ine / division-territorial-censal, subdere / division-territorial
   3 datasets: deis / establecimientos-salud, deis / establecimientos-salud-historial, deis / resumen-estadistico-mensual
   3 datasets: deis / codigos-rem, deis / codigos-rem-col-numericas, deis / codigos-rem-prestaciones
   1 datasets: subdere / division-territorial-cambios
   1 datasets: subdere / division-territorial v2
   1 datasets: red-movilidad / posicion-diaria
   1 dataset

En el estado *hoy* hay datasets aislados (sin ninguna relación declarada con otro) y varios grupos separados; en el estado
*objetivo* quedan solo dos grupos: salud, censo, SINCA y territorio por un lado, y el transporte (GTFS más posiciones) por otro.
Lo que los une no son las tablas sino el espacio: ver la capa espacial.

---
## 3. El grafo interactivo

* **Estado** y **capas** arriba; los grupos y el buscador filtran los nodos.
* Clic en un nodo: columnas de unión, quién lo referencia, con quién comparte tabla.
* **Tabla**: las mismas relaciones en texto (equivalente accesible del dibujo).
* El estado se guarda en la URL (`#n=<id>&estado=hoy&par=1`), así se puede compartir una vista.

El visor es un HTML autocontenido; además de mostrarse aquí se guarda en un archivo para abrirlo en el navegador.

In [4]:
destino = pathlib.Path(tempfile.gettempdir()) / "grafo-datopia.html"
gd.escribir_html(grafo, "visor.html", destino)
print(f"Visor guardado en {destino} (ábrelo en el navegador para verlo a pantalla completa)")
display(HTML(destino.read_text(encoding="utf-8")))

Visor guardado en /tmp/grafo-datopia.html (ábrelo en el navegador para verlo a pantalla completa)


---
## 4. Las relaciones que aún no están declaradas

Estas relaciones salen de `relaciones-objetivo.json`. Las marcadas con ⚠ tienen un problema conocido: **la relación existe**,
pero parte de las filas no cruza hoy, y se dibujan en ámbar.

In [5]:
import pandas as pd

nombre = {n["id"]: n["etiqueta"] for n in grafo["nodos"]}
filas = [{
    "desde": nombre[e["desde"]], "hacia": nombre[e["hacia"]],
    "columnas": ", ".join(e["columnas_desde"]) + " → " + ", ".join(e["columnas_hacia"]),
    "hoy": ", ".join(e["columnas_desde_hoy"]) if e["columnas_desde_hoy"] != e["columnas_desde"] else "",
    "issues": " ".join(e["issues"]),
    "⚠": e["advertencia"] or "",
} for e in grafo["aristas"] if e["tipo"] == "tabular" and e["estado"] == "objetivo"]
pd.set_option("display.max_colwidth", 90)
pd.DataFrame(filas)

,desde,hacia,columnas,hoy,issues,⚠
0,deis / establecimientos-salud,subdere / division-territorial,cut_com → cut_com,cod_comuna,#56 #59,
1,sinca / estaciones,subdere / division-territorial,cut_com → cut_com,,#57 #59,
2,subdere / division-territorial-cambios,subdere / division-territorial,"cut_anterior, cut_nuevo → cut_com",,#59,
3,deis / resumen-estadistico-mensual,deis / codigos-rem,"year, tipo_serie, cod_prestacion, cod_col_numerica → year, tipo_serie, cod_prestacion,...",,#46,En la serie BS (2017-2026) cod_prestacion pierde el cero inicial en los hechos y no cr...
4,ine / censos-poblacionales v1l,ine / preguntas-respuestas-censo,"year, tipo_registro, pregunta, respuesta → year, tipo_registro, num_pregunta, num_resp...",,#47,El emparejamiento exacto de columnas (pregunta/respuesta contra num_pregunta/num_respu...
5,dtpm / gtfs-stop-times,dtpm / gtfs-stops,"feed_id, stop_id → feed_id, stop_id",,,
6,dtpm / gtfs-pathways,dtpm / gtfs-stops,"feed_id, from_stop_id, to_stop_id → feed_id, stop_id",,,
7,dtpm / gtfs-transfers,dtpm / gtfs-stops,"feed_id, from_stop_id, to_stop_id → feed_id, stop_id",,,
8,dtpm / gtfs-trips,dtpm / gtfs-calendar,"feed_id, service_id → feed_id, service_id",,,
9,sinca / estaciones-variables,sinca / estaciones,estacion_id → estacion_id,,,


La columna *hoy* muestra cómo se llama hoy la columna que el objetivo renombra (#56: `cod_comuna` pasa a `cut_com`). Algunas
columnas del objetivo todavía no existen en los archivos (por ejemplo `cut_com` en las estaciones SINCA); el visor lo indica
en la tarjeta del nodo como *columna aún no publicada*.

Una relación sobre el territorio vale **a una fecha**: la división territorial cambia (`snapshot_date`), y cada dataset tiene su
propio tiempo (año del censo, `fecha_inicio` de la estación, `feed_start_date` del GTFS). Cada arista lo anota como *vigencia*.

---
## 5. Un cruce espacial real (opcional, requiere cuenta)

La arista espacial *estaciones SINCA → comunas* dice que cada estación cae dentro de una comuna. Hoy eso se puede comprobar con
los archivos publicados: un cruce punto-en-polígono con DuckDB contra la última foto de la división territorial. Es el mismo
cruce con el que se llenará `cut_com` en las estaciones (#57).

Necesita credenciales (`config.json`, o el email y la contraseña en Colab). Sin ellas, la celda lo avisa y sigue.

In [6]:
# @title Cruce punto-en-polígono: estaciones SINCA dentro de comunas
import getpass, unicodedata
import duckdb, requests

EN_COLAB = "google.colab" in sys.modules
URL_API = "https://api.datopia.dataobservatory.net/api"


def credenciales():
    if EN_COLAB:
        return input("Email: ").strip(), getpass.getpass("Contraseña: ")
    ruta = pathlib.Path("../../../config.json")
    if not ruta.exists():
        return None, None
    cfg = json.loads(ruta.read_text())
    usuario = cfg.get("test_user", {})
    return usuario.get("email"), usuario.get("password")


EMAIL, PASSWORD = credenciales()
if not EMAIL or not PASSWORD:
    print("Sin credenciales: se omite esta sección (copia config.example.json a config.json en la raíz del repo).")
else:
    token = requests.post(f"{URL_API}/auth/login", json={"email": EMAIL, "password": PASSWORD}, timeout=30).json()["id_token"]
    creds = requests.post(f"{URL_API}/auth/session/s3", headers={"Authorization": f"Bearer {token}"}, timeout=30).json()
    con = duckdb.connect()
    con.execute("INSTALL httpfs; LOAD httpfs")
    con.execute("INSTALL spatial; LOAD spatial")
    con.execute(f"""
        CREATE OR REPLACE SECRET s3 (TYPE S3, KEY_ID '{creds["access_key_id"]}', SECRET '{creds["secret_access_key"]}',
                                     SESSION_TOKEN '{creds["session_token"]}', REGION '{creds["region"]}')
    """)
    prefijo = {d["id"]: d["s3_prefix"] for d in datamodel["datasets"]}
    base = f"s3://{creds['bucket']}/"
    estaciones = base + prefijo["medio_fisico_cl_sinca_estaciones_v1"]
    comunas = base + prefijo["geografia_social_cl_subdere_division_territorial_v2"]

    foto = con.execute(f"SELECT max(snapshot_date) FROM read_parquet('{comunas}snapshot_date=*/*.parquet', hive_partitioning=true)").fetchone()[0]
    print(f"Foto de la división territorial: snapshot_date = {foto}")

    cruce = con.execute(f"""
        WITH est AS (
            SELECT estacion_id, nombre, comuna AS comuna_declarada, geometry
            FROM read_parquet('{estaciones}*.parquet')
            WHERE fecha_fin IS NULL AND geometry IS NOT NULL
        ), com AS (
            SELECT cut_com, nom_com, geometry
            FROM read_parquet('{comunas}snapshot_date=*/*.parquet', hive_partitioning=true)
            WHERE snapshot_date = '{foto}' AND geometry IS NOT NULL
        )
        SELECT est.estacion_id, est.nombre, est.comuna_declarada, com.cut_com, com.nom_com AS comuna_cruce
        FROM est LEFT JOIN com ON ST_Within(est.geometry, com.geometry)
        ORDER BY est.estacion_id
    """).df()

    def norma(t):
        t = unicodedata.normalize("NFKD", str(t)).encode("ascii", "ignore").decode().lower().strip()
        return t

    cruce["coincide"] = [norma(a) == norma(b) for a, b in zip(cruce["comuna_declarada"], cruce["comuna_cruce"])]
    print(f"{len(cruce)} estaciones con geometría vigente · {cruce['cut_com'].notna().sum()} caen dentro de una comuna · "
          f"{int(cruce['coincide'].sum())} con el mismo nombre que declara la estación")
    display(cruce[~cruce["coincide"]].head(15))

Foto de la división territorial: snapshot_date = 2018-09-06


224 estaciones con geometría vigente · 223 caen dentro de una comuna · 212 con el mismo nombre que declara la estación


,estacion_id,nombre,comuna_declarada,cut_com,comuna_cruce,coincide
7,207,Estación Ferrocarriles,Mejillones,NaN,NaN,False
9,210,Estación Hospital,María Elena,02201,Calama,False
60,402,Estación Chacay,Los Vilos,04204,Salamanca,False
72,425,Estación La Serena,Coquimbo,04101,La Serena,False
84,512,Estación Las Gaviotas,Concón,05107,Quintero,False
92,523,Estación Romeral,Catemu,05503,Hijuelas,False
98,534,"Estación La Cruz, Colbún",Calera,05504,La Cruz,False
169,859,Estación Lota rural,Coronel,08106,Lota,False
170,860,Estación Lota urbana,Coronel,08106,Lota,False
191,A04,Estación Entre Lagos,Osorno,10304,Puyehue,False


Las estaciones que no coinciden son justo las que hoy se unirían mal por nombre (tildes, mayúsculas, comunas renombradas): con
`cut_com` en la tabla, esa unión pasa a ser por código.

---
## 6. Cómo mantener el grafo

* Cuando una relación objetivo se publica y entra al modelo público, **sale sola** del archivo de objetivo: basta borrarla de
  `relaciones-objetivo.json`. Aparece como relación declarada (línea continua) y el visor deja de marcarla como pendiente.
* Cuando #59 defina la llave de la tabla de territorio (tabla actual plana, o una foto por fecha), solo cambia el id de destino
  en `relaciones-objetivo.json` (`tabla_geografica`): las aristas hacia el territorio lo siguen.
* El estado de cada arista se calcula de las columnas reales del modelo público; una columna que falta se marca como pendiente.